# World Models, JEPA and Recursive Self-Improvement: Three Bets Beyond the Next Token

The full code of [World Models, JEPA and Recursive Self-Improvement: Three Bets Beyond the Next Token](https://residual-stream.lvtan71.workers.dev/posts/world-models-jepa-and-self-improvement/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/world-models-jepa-and-self-improvement) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/world-models-jepa-and-self-improvement.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "world-models-jepa-and-self-improvement"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `pendulum_imagination.py`

A small network learns one step of a damped pendulum from 20,000 random transitions, then imagines 1 to 100 steps ahead on its own predictions; the angle error against the real pendulum, 1,000 start states, 3 seeds.

In [ ]:
!{sys.executable} pendulum_imagination.py

The post printed:

```text
error of imagined angles against the real ones (radians; a random guess would be
off by about 1.8 root-mean-square), 1,000 start states, 3 seeds
        steps imagined        1        5       15       50      100
  RMS error, seed mean   0.0029   0.0148   0.0493   0.1205   0.2679
median error, seed mean   0.0023   0.0114   0.0387   0.0710   0.1448
   RMS, per seed at 15   0.0247   0.0645   0.0586
```

## `dreamer_budget.py`

How many steps one Dreamer-style update imagines at DreamerV3's default batch and horizon, and how long the same number of real steps would take. Plain arithmetic.

In [ ]:
!{sys.executable} dreamer_budget.py

The post printed:

```text
per update: 1,024 start states from replay, 15,360 imagined steps,
computed as 15 sequential batched steps of 1,024 states each
  15,360 real steps in a fast simulator (1 ms/step): 15 s
  15,360 real steps in a robot acting at 20 Hz (50 ms/step): 768 s
```

## `jepa_collapse.py`

Joint-embedding prediction on a toy world (8-number hidden state rendered into noisy, masked 64-number views): an untrained encoder, a shared encoder with no tricks, stop-gradient, an EMA target with and without a predictor, and VICReg regularizers, 3 seeds each; final loss, embedding spread and a linear probe's R^2.

In [ ]:
!{sys.executable} jepa_collapse.py

The post printed:

```text
variant                      final loss  embedding std  probe R^2   (mean of 3 seeds, 10,000 steps)
untrained encoder                0.3205       0.055590       0.33
shared encoder, no tricks        0.0000       0.000174       0.07
stop-gradient                    1.3830       0.253997       0.61
EMA target + predictor           1.9868       0.306188       0.61
EMA, no predictor                0.0000       0.000054      -0.00
VICReg regularizers             15.7369       0.927692       0.61
```